# 07 | LangGraph 条件边：让流程根据 State 分支

第 06 篇里的流程是固定的：

```text
START -> clean_text -> count_chars -> END
```

但真实流程经常不是一条直线。比如用户提交一个问题后，技术问题、账单问题、普通问题应该进入不同处理节点。

这就是 `conditional edge` 要解决的问题：**根据当前 State 的内容，决定下一步去哪个 Node。**

## 一、从一个分流流程开始

下面这张图会做三件事：

```text
输入用户问题
 -> classify_question 判断问题类型
 -> 条件边根据 category 选择下一个节点
 -> 生成不同回复
```

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


# Category：限制 category 只能是这三个路由值之一。
Category = Literal["technical", "billing", "general"]


# State：保存用户问题、分类结果和最终回复。
class QuestionState(TypedDict):
    question: str
    category: Category
    answer: str


# Node：读取 question，判断问题类型，并把结果写入 category。
def classify_question(state: QuestionState) -> dict[str, Category]:
    question = state["question"]

    if any(word in question for word in ["登录", "报错", "接口"]):
        return {"category": "technical"}
    if any(word in question for word in ["退款", "发票", "扣费"]):
        return {"category": "billing"}
    return {"category": "general"}


# Router function：读取 classify_question 写入的 category，决定下一步分支。
# 它只返回路由值，不负责写入 answer。
def route_by_category(state: QuestionState) -> Category:
    return state["category"]


# Branch node：技术问题分支，写入技术类回复。
def technical_answer(state: QuestionState) -> dict[str, str]:
    return {"answer": "这是技术问题，请先提供报错截图和操作步骤。"}


# Branch node：账单问题分支，写入账单类回复。
def billing_answer(state: QuestionState) -> dict[str, str]:
    return {"answer": "这是账单问题，请提供订单号或发票抬头。"}


# Branch node：普通问题分支，写入通用回复。
def general_answer(state: QuestionState) -> dict[str, str]:
    return {"answer": "这是普通问题，我们会继续帮你确认。"}


# StateGraph：用 QuestionState 作为整张图的状态结构。
builder = StateGraph(QuestionState)

# add_node：先把分类节点和三个分支节点注册进图。
builder.add_node("classify_question", classify_question)
builder.add_node("technical_answer", technical_answer)
builder.add_node("billing_answer", billing_answer)
builder.add_node("general_answer", general_answer)

# START：图从 classify_question 开始执行。
builder.add_edge(START, "classify_question")

# conditional edge：classify_question 执行后，调用 route_by_category 决定下一步。
builder.add_conditional_edges(
    "classify_question",
    route_by_category,
    {
        # mapping：router 返回 technical，就进入 technical_answer 节点。
        "technical": "technical_answer",
        # mapping：router 返回 billing，就进入 billing_answer 节点。
        "billing": "billing_answer",
        # mapping：router 返回 general，就进入 general_answer 节点。
        "general": "general_answer",
    },
)

# 三个分支节点都是终点：写完 answer 后流程结束。
builder.add_edge("technical_answer", END)
builder.add_edge("billing_answer", END)
builder.add_edge("general_answer", END)

# compile：把构建器编译成可运行的 Graph。
graph = builder.compile()

# invoke：传入初始 State，运行图并得到最终 State。
graph.invoke(
    {
        "question": "控制台登录一直报错，怎么办？",
        # 初始 category 只是占位，真正结果会由 classify_question 覆盖。
        "category": "general",
        "answer": "",
    }
)

这段代码里，最重要的是这三层：

```text
classify_question
  负责写入 category

route_by_category
  负责读取 category，返回路由值

add_conditional_edges
  负责把路由值映射到下一个节点
```

## 二、条件边由哪几部分组成

普通边是固定路线：

```python
builder.add_edge("node_a", "node_b")
```

它的意思是：`node_a` 执行完，一定进入 `node_b`。

条件边是动态路线：

```python
builder.add_conditional_edges("classify_question", route_by_category, mapping)
```

它的意思是：`classify_question` 执行完，先调用 `route_by_category`，再根据返回值选择下一个节点。

### Router function 是什么

`router function` 是一个普通 Python 函数。它接收当前 State，返回一个路由值。

```python
def route_by_category(state: QuestionState) -> Category:
    return state["category"]
```

这个函数不负责生成回答，也不负责更新 State。它只回答一个问题：

```text
下一步应该走哪条分支？
```

在这个例子里，它可能返回：

```text
technical
billing
general
```

### Mapping 是什么

`add_conditional_edges` 的第三个参数是一个映射表，用来把 router 的返回值翻译成节点名。

```python
{
    "technical": "technical_answer",
    "billing": "billing_answer",
    "general": "general_answer",
}
```

可以这样读：

| router 返回值 | 下一步节点 |
| --- | --- |
| `technical` | `technical_answer` |
| `billing` | `billing_answer` |
| `general` | `general_answer` |

官方文档也说明，条件边会在某个节点执行后调用路由函数；路由函数接收当前 State 并返回一个值。这个值可以直接当作下一个节点名，也可以通过字典映射到节点名。

## 三、看三种输入怎么分支

同一张图，换不同输入，会走不同分支。

In [ ]:
examples = [
    "控制台登录一直报错，怎么办？",
    "我需要修改发票抬头。",
    "你们的客服时间是几点？",
]

for question in examples:
    result = graph.invoke(
        {
            "question": question,
            "category": "general",
            "answer": "",
        }
    )
    print(result["category"], "->", result["answer"])

执行过程可以理解成：

```text
question
 -> classify_question 写入 category
 -> route_by_category 读取 category
 -> add_conditional_edges 根据 category 找到下一个节点
 -> 对应 answer 节点写入 answer
 -> END
```

## 四、画成图会更清楚

自动生成图可以这样看：

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## 五、写条件边时的判断标准

可以先记住三个小规则：

| 问题 | 建议 |
| --- | --- |
| 路线永远固定？ | 用 `add_edge` |
| 路线要根据 State 判断？ | 用 `add_conditional_edges` |
| 同一个节点后面又写普通边又写条件边？ | 尽量避免，容易让流程变得难推理 |

### 小结

`conditional edge` 可以用一句话理解：

```text
节点执行完以后，调用一个 router function；
router 根据 State 返回路由值；
LangGraph 根据这个值决定下一步去哪个节点。
```

如果第 06 篇解决的是“图如何按固定顺序执行”，这一篇解决的就是“图如何根据结果选择下一步”。